# Gradient Boosting

In [1]:
import pandas as pd

data = pd.read_csv("./Training_Data.csv")
data.describe()

,number_of_maps,firepower,entrying,trading,opening,clutching,sniping,utility,igl
count,175.000000,175.000000,175.000000,175.000000,175.000000,175.000000,175.000000,175.000000,175.000000
mean,50.108571,54.708571,48.554286,47.342857,53.548571,50.885714,20.542857,58.771429,0.194286
std,16.175925,24.174668,25.217586,20.248943,22.222146,17.346985,37.558493,18.594409,0.396785
min,14.000000,2.000000,5.000000,6.000000,5.000000,18.000000,0.000000,17.000000,0.000000
25%,37.000000,35.000000,29.500000,31.500000,35.000000,38.000000,0.000000,45.500000,0.000000
50%,51.000000,55.000000,47.000000,47.000000,52.000000,48.000000,0.000000,58.000000,0.000000
75%,58.000000,73.500000,69.000000,62.000000,75.000000,65.000000,3.500000,72.500000,0.000000
max,106.000000,100.000000,98.000000,96.000000,97.000000,91.000000,95.000000,95.000000,1.000000


In [2]:
feature_cols = [
    "firepower",
    "entrying",
    "trading",
    "opening",
    "clutching",
    "sniping",
    "utility",
    "igl",
]

x = data[feature_cols]
y = data["position"]

# can comment line below out
y.replace("Linchpin", "Closer", inplace=True)

y.head()
y.describe()

count        175
unique         3
top       Closer
freq          72
Name: position, dtype: object

In [3]:
import numpy as np
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    ConfusionMatrixDisplay,
)
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from scipy.stats import randint, uniform


x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.25, random_state=42, stratify=y
)

In [4]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.fit_transform(y_test)

model = XGBClassifier(
    n_estimators=600, max_depth=5, learning_rate=0.1, objective="multi:logistic"
)
model.fit(x_train, y_train)

,objective,'multi:softprob'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [5]:
y_pred = model.predict(x_test)

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy: ", accuracy)

Accuracy:  0.8181818181818182


In [6]:
param_dist = {
    "n_estimators": randint(100, 600),
    "max_depth": randint(3, 8),
    "learning_rate": uniform(loc=0.01, scale=0.3),
    "subsample": uniform(0.7, 0.3),
    "colsample_bytree": uniform(0.7, 0.3),
}


gb = XGBClassifier(random_state=42, n_jobs=1, objective="multi:softprob")

rand_search = RandomizedSearchCV(
    gb,
    param_distributions=param_dist,
    n_iter=5,
    cv=5,
    scoring="accuracy",
    n_jobs=2,
    random_state=42,
).fit(x_train, y_train)

best_rf = rand_search.best_estimator_

params = rand_search.best_params_

print("Best hyperparameters:", params)

Best hyperparameters: {'colsample_bytree': np.float64(0.981565812704725), 'learning_rate': np.float64(0.010233629752304298), 'max_depth': 6, 'n_estimators': 376, 'subsample': np.float64(0.8852444528883149)}


In [7]:
opt_model = XGBClassifier(**params, objective="multi:softprob")
opt_model.fit(x_train, y_train)

,objective,'multi:softprob'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,np.float64(0.981565812704725)
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [8]:
y_pred = opt_model.predict(x_test)

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy: ", accuracy)

Accuracy:  0.8409090909090909


# Separate classifier for each position, merging closer and linchpin

In [9]:
y = data["position"]
y.replace("AWPer", 1, inplace=True)
y.replace("Closer", 0, inplace=True)
y.replace("Linchpin", 0, inplace=True)
y.replace("Opener", 0, inplace=True)

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.25, random_state=42, stratify=y
)

y_train = le.fit_transform(y_train)
y_test = le.fit_transform(y_test)

param_dist = {
    "n_estimators": randint(100, 600),
    "max_depth": randint(3, 8),
    "learning_rate": uniform(loc=0.01, scale=0.3),
    "subsample": uniform(0.7, 0.3),
    "colsample_bytree": uniform(0.7, 0.3),
}


gb = XGBClassifier(random_state=42, n_jobs=1, objective="binary:logistic")

rand_search = RandomizedSearchCV(
    gb,
    param_distributions=param_dist,
    n_iter=5,
    cv=5,
    scoring="accuracy",
    n_jobs=2,
    random_state=42,
).fit(x_train, y_train)

best_rf = rand_search.best_estimator_

params = rand_search.best_params_

print("Best hyperparameters:", params)

opt_model = XGBClassifier(**params, objective="binary:logistic")
opt_model.fit(x_train, y_train)

y_pred = opt_model.predict(x_test)

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy: ", accuracy)

/tmp/ipykernel_2011/1262589600.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y.replace("Opener", 0, inplace=True)


Best hyperparameters: {'colsample_bytree': np.float64(0.8123620356542087), 'learning_rate': np.float64(0.2952142919229748), 'max_depth': 5, 'n_estimators': 171, 'subsample': np.float64(0.8795975452591109)}
Accuracy:  1.0


In [10]:
y = data["position"]
y.replace("AWPer", 0, inplace=True)
y.replace("Closer", 1, inplace=True)
y.replace("Linchpin", 1, inplace=True)
y.replace("Opener", 0, inplace=True)

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.25, random_state=42, stratify=y
)

y_train = le.fit_transform(y_train)
y_test = le.fit_transform(y_test)

param_dist = {
    "n_estimators": randint(100, 600),
    "max_depth": randint(3, 8),
    "learning_rate": uniform(loc=0.01, scale=0.3),
    "subsample": uniform(0.7, 0.3),
    "colsample_bytree": uniform(0.7, 0.3),
}


gb = XGBClassifier(random_state=42, n_jobs=1, objective="binary:logistic")

rand_search = RandomizedSearchCV(
    gb,
    param_distributions=param_dist,
    n_iter=5,
    cv=5,
    scoring="accuracy",
    n_jobs=2,
    random_state=42,
).fit(x_train, y_train)

best_rf = rand_search.best_estimator_

params = rand_search.best_params_

print("Best hyperparameters:", params)

opt_model = XGBClassifier(**params, objective="binary:logistic")
opt_model.fit(x_train, y_train)

y_pred = opt_model.predict(x_test)

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy: ", accuracy)

Best hyperparameters: {'colsample_bytree': np.float64(0.8123620356542087), 'learning_rate': np.float64(0.2952142919229748), 'max_depth': 5, 'n_estimators': 171, 'subsample': np.float64(0.8795975452591109)}
Accuracy:  1.0


In [11]:
y = data["position"]
y.replace("AWPer", "Not", inplace=True)
y.replace("Closer", "Not", inplace=True)
y.replace("Linchpin", "Not", inplace=True)

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.25, random_state=42, stratify=y
)

param_dist = {
    "n_estimators": randint(100, 600),
    "max_depth": randint(3, 8),
    "learning_rate": uniform(loc=0.01, scale=0.3),
    "subsample": uniform(0.7, 0.3),
    "colsample_bytree": uniform(0.7, 0.3),
}


gb = XGBClassifier(random_state=42, n_jobs=1, objective="binary:logistic")

rand_search = RandomizedSearchCV(
    gb,
    param_distributions=param_dist,
    n_iter=5,
    cv=5,
    scoring="accuracy",
    n_jobs=2,
    random_state=42,
).fit(x_train, y_train)

best_rf = rand_search.best_estimator_

params = rand_search.best_params_

print("Best hyperparameters:", params)

opt_model = XGBClassifier(**params, objective="binary:logistic")
opt_model.fit(x_train, y_train)

y_pred = opt_model.predict(x_test)

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy: ", accuracy)

Best hyperparameters: {'colsample_bytree': np.float64(0.8123620356542087), 'learning_rate': np.float64(0.2952142919229748), 'max_depth': 5, 'n_estimators': 171, 'subsample': np.float64(0.8795975452591109)}
Accuracy:  1.0


In [15]:
print(y_test)
print(y_pred)

66     0
47     0
110    0
148    0
103    0
130    0
161    0
64     1
41     0
20     1
135    0
37     0
156    0
174    0
171    1
158    0
170    1
167    0
18     0
131    0
137    0
90     0
112    0
2      0
72     0
75     0
80     0
93     0
126    0
108    1
150    0
1      0
119    1
95     0
117    0
27     0
4      1
151    0
10     1
29     0
26     0
152    1
160    1
109    0
Name: position, dtype: int64
[0 0 0 0 0 0 0 1 0 1 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 0 0 1
 0 1 0 0 1 1 0]


In [13]:
print(x_test)

     firepower  entrying  trading  opening  clutching  sniping  utility  igl
66          28        18       28       34         45        1       46    0
47          81        72       55       82         38        0       50    0
110         47        32       43       27         65        5       62    0
148         19        15       42       42         84        0       66    0
103        100        84       71       97         42        1       35    0
130          2        77        6       27         21        0       74    1
161         95        58       60       76         70        1       57    0
64          66        11       32       52         78       93       60    0
41          65        46       62       63         52        1       39    0
20          30        29       84       31         89       89       58    0
135         79        53       37       89         26        0       56    0
37          49        62       48       51         54        0       70    1

In [14]:
p = [9, 100, 7, 100, 39, 1, 28, 0]
player = pd.DataFrame([p])
player.columns = feature_cols
print(opt_model.predict(player))

[0]


In [17]:
y = data["position"]
print(y.head())
y.replace("AWPer", "Not", inplace=True)
y.replace("Closer", "Not", inplace=True)
y.replace("Linchpin", "Not", inplace=True)

print(y.head())

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.25, random_state=42, stratify=y
)

print(y_test)

param_dist = {
    "n_estimators": randint(100, 600),
    "max_depth": randint(3, 8),
    "learning_rate": uniform(loc=0.01, scale=0.3),
    "subsample": uniform(0.7, 0.3),
    "colsample_bytree": uniform(0.7, 0.3),
}


gb = XGBClassifier(random_state=42, n_jobs=1, objective="binary:logistic")

rand_search = RandomizedSearchCV(
    gb,
    param_distributions=param_dist,
    n_iter=5,
    cv=5,
    scoring="accuracy",
    n_jobs=2,
    random_state=42,
).fit(x_train, y_train)

best_rf = rand_search.best_estimator_

params = rand_search.best_params_

print("Best hyperparameters:", params)

opt_model = XGBClassifier(**params, objective="binary:logistic")
opt_model.fit(x_train, y_train)

y_pred = opt_model.predict(x_test)

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy: ", accuracy)

0    0
1    0
2    0
3    0
4    1
Name: position, dtype: int64
0    0
1    0
2    0
3    0
4    1
Name: position, dtype: int64
66     0
47     0
110    0
148    0
103    0
130    0
161    0
64     1
41     0
20     1
135    0
37     0
156    0
174    0
171    1
158    0
170    1
167    0
18     0
131    0
137    0
90     0
112    0
2      0
72     0
75     0
80     0
93     0
126    0
108    1
150    0
1      0
119    1
95     0
117    0
27     0
4      1
151    0
10     1
29     0
26     0
152    1
160    1
109    0
Name: position, dtype: int64
Best hyperparameters: {'colsample_bytree': np.float64(0.8123620356542087), 'learning_rate': np.float64(0.2952142919229748), 'max_depth': 5, 'n_estimators': 171, 'subsample': np.float64(0.8795975452591109)}
Accuracy:  1.0


In [18]:
data.head()

,name,position,number_of_maps,firepower,entrying,trading,opening,clutching,sniping,utility,igl
0,alex666,0,83,27,44,43,26,58,0,61,1
1,npl,0,83,85,44,60,76,47,0,48,0
2,kensizor,0,83,70,94,56,63,37,0,41,0
3,esenthial,0,83,31,40,79,22,61,0,23,0
4,s1zzi,1,83,57,47,32,77,63,93,33,0
